# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 12

Version 11 introduced the public **WikiText-103 Raw v1** dataset, a BPE tokenizer trained directly in the notebook and a small two-block decoder-only Transformer.

Version 12 keeps the same data selection, tokenizer procedure, model architecture, causal objective, batch size, learning rate and 20-epoch training budget.

The new concept is **training state and checkpoint management**.

The progression is:

text from WikiText-103  
→ BPE subword tokens  
→ fixed-length token sequences  
→ decoder-only Transformer  
→ training to epoch 10  
→ last checkpoint for resume  
→ restored training state  
→ training to epoch 20  
→ best checkpoint for model selection  
→ restored best model  
→ final test evaluation

Training data updates model parameters, validation data selects the best checkpoint, and the test split is evaluated only after model selection is complete.

Version 12 therefore keeps the Version 11 language-model experiment intact while making training resumable and model selection explicit.

## 1. Imports and Configuration

Version 12 keeps the Version 11 data selection, tokenizer configuration, model architecture and optimization settings.

The experiment still uses:

- 10,000 training rows, 1,000 validation rows and 1,000 test rows;
- a BPE vocabulary of 2,048 tokens;
- contexts of 16 tokens;
- 12,000 training sequences, 2,000 validation sequences and 2,000 test sequences;
- batch size 32;
- learning rate 1.0;
- the same two-block decoder-only Transformer.

`INTERRUPT_AFTER_EPOCH` defines the point where training is intentionally interrupted.

`TOTAL_EPOCHS` preserves the 20-epoch training budget of Version 11.

Training examples use an epoch-specific shuffle seed so that a restored model resumes with the shuffle associated with the correct next epoch.

The new concept in this version is checkpoint management rather than a change to the language-model architecture.

In [1]:
import os
import shutil
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["RAYON_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import tensorflow as tf

from tokenizers import Tokenizer
from tokenizers.decoders import BPEDecoder
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [2]:
SEED = 42

TRAIN_ROWS = 10000
VALIDATION_ROWS = 1000
TEST_ROWS = 1000

VOCABULARY_SIZE = 2048
CONTEXT_LENGTH = 16

MAX_TRAIN_SEQUENCES = 12000
MAX_VALIDATION_SEQUENCES = 2000
MAX_TEST_SEQUENCES = 2000

BATCH_SIZE = 32
LEARNING_RATE = 1.0

INTERRUPT_AFTER_EPOCH = 10
TOTAL_EPOCHS = 20

EMBEDDING_DIM = 8
ATTENTION_DIM = 16
FEED_FORWARD_DIM = 16
NUM_BLOCKS = 2
LAYER_NORM_EPSILON = 1e-5

tf.keras.utils.set_random_seed(SEED)

In [3]:
DATASET_DIRECTORY = "/kaggle/input/datasets/lucalullo/wiki-text-103-raw-v1"

train_files = [
    os.path.join(DATASET_DIRECTORY, "train-00000-of-00002.parquet"),
    os.path.join(DATASET_DIRECTORY, "train-00001-of-00002.parquet"),
]

validation_file = os.path.join(DATASET_DIRECTORY, "validation-00000-of-00001.parquet")
test_file = os.path.join(DATASET_DIRECTORY, "test-00000-of-00001.parquet")

for path in train_files + [validation_file, test_file]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

print("Dataset files found:")
for path in train_files + [validation_file, test_file]:
    print(os.path.basename(path))

Dataset files found:
train-00000-of-00002.parquet
train-00001-of-00002.parquet
validation-00000-of-00001.parquet
test-00000-of-00001.parquet


## 2. Public Dataset

Version 12 keeps the **WikiText-103 Raw v1** dataset introduced in Version 11.

The dataset is attached to the Kaggle notebook as an input and contains the original training, validation and test splits in Parquet format.

The files are:

- two Parquet shards for the training split;
- one Parquet file for the validation split;
- one Parquet file for the test split.

Each file contains a single `text` column.

The original split boundaries remain unchanged. Version 12 does not create a new random train/validation/test split.

To preserve direct comparability with Version 11, the same reproducible subset is selected independently from each official split:

- 10,000 training rows;
- 1,000 validation rows;
- 1,000 test rows.

Empty rows are removed before sampling because WikiText contains blank entries that represent document and section boundaries rather than useful standalone training text.

The dataset path was validated before this section. If the external Kaggle input is unavailable, the notebook stops explicitly rather than downloading or substituting another corpus.

The dataset language is English. Source, provenance and licensing information remain those documented for the attached `wikitext-103-raw-v1` dataset.

In [4]:
train_data = pd.concat(
    [pd.read_parquet(path, columns=["text"]) for path in train_files],
    ignore_index=True,
)

validation_data = pd.read_parquet(validation_file, columns=["text"])
test_data = pd.read_parquet(test_file, columns=["text"])

print("Original rows:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Original rows:
Train: 1801350
Validation: 3760
Test: 4358


In [5]:
def prepare_text_subset(data, number_of_rows, seed):
    cleaned = data.dropna(subset=["text"]).copy()
    cleaned = cleaned[cleaned["text"].str.strip().str.len() > 0]
    return cleaned.sample(n=min(number_of_rows, len(cleaned)), random_state=seed).reset_index(drop=True)


train_data = prepare_text_subset(train_data, TRAIN_ROWS, SEED)
validation_data = prepare_text_subset(validation_data, VALIDATION_ROWS, SEED)
test_data = prepare_text_subset(test_data, TEST_ROWS, SEED)

print("Experiment rows:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

print("\nExample training text:")
print(train_data.loc[0, "text"])

Experiment rows:
Train: 10000
Validation: 1000
Test: 1000

Example training text:
 In cooperation with various nonprofit organizations , since 2008 the company has organized the THSRC Smile Program , which offers minority groups and families the chance to experience the comfort and convenience of high @-@ speed rail rides . THSRC Smile Program gave its first ride on 23 January 2008 to children , social workers and parents from the Fushan branch of the Taiwan Fund for Children and Families . As of the end of 2011 , 12 @,@ 231 people in total , comprising 229 underprivileged groups and families , have participated in the THSRC Smile Program . 



## 3. Subword Tokenizer

Version 12 keeps the **Byte Pair Encoding (BPE)** tokenizer introduced in Version 11.

BPE starts from small text units and repeatedly merges frequent neighboring units to build a vocabulary of reusable subword tokens.

The tokenizer is trained only on the selected training subset. Validation and test text are never used to construct the vocabulary, preserving the separation between the official dataset splits.

The tokenizer configuration remains unchanged from Version 11:

- a maximum vocabulary size of 2,048 tokens;
- `[UNK]` as the special unknown token;
- whitespace pre-tokenization;
- a minimum token frequency of 2;
- `</w>` as the end-of-word suffix.

Keeping the same tokenizer procedure ensures that Version 12 changes the training organization rather than the language-model representation.

After training, the tokenizer converts text into integer token identifiers that can be processed by the same decoder-only Transformer architecture used in Version 11.

In [6]:
tokenizer = Tokenizer(BPE(
    unk_token="[UNK]",
    end_of_word_suffix="</w>"
))

tokenizer.pre_tokenizer = Whitespace()
tokenizer.decoder = BPEDecoder(suffix="</w>")

trainer = BpeTrainer(
    vocab_size=VOCABULARY_SIZE,
    min_frequency=2,
    special_tokens=["[UNK]"],
    end_of_word_suffix="</w>"
)

tokenizer.train_from_iterator(train_data["text"], trainer=trainer)

vocabulary_size = tokenizer.get_vocab_size()

print("Vocabulary size:", vocabulary_size)




Vocabulary size: 2048


### Inspecting the Learned Tokens

The tokenizer can now represent text using variable-length subword units instead of individual characters.

Frequent words or word fragments may become single tokens, while less frequent words can be decomposed into multiple smaller tokens.

In [7]:
example_text = train_data.loc[0, "text"]
encoded_example = tokenizer.encode(example_text)

print("Text:")
print(example_text)

print("\nTokens:")
print(encoded_example.tokens[:30])

print("\nToken IDs:")
print(encoded_example.ids[:30])

print("\nNumber of characters:", len(example_text))
print("Number of tokens:", len(encoded_example.ids))

Text:
 In cooperation with various nonprofit organizations , since 2008 the company has organized the THSRC Smile Program , which offers minority groups and families the chance to experience the comfort and convenience of high @-@ speed rail rides . THSRC Smile Program gave its first ride on 23 January 2008 to children , social workers and parents from the Fushan branch of the Taiwan Fund for Children and Families . As of the end of 2011 , 12 @,@ 231 people in total , comprising 229 underprivileged groups and families , have participated in the THSRC Smile Program . 


Tokens:
['In</w>', 'co', 'oper', 'ation</w>', 'with</w>', 'vari', 'ous</w>', 'n', 'on', 'pro', 'f', 'it</w>', 'org', 'ani', 'z', 'ations</w>', ',</w>', 'since</w>', '200', '8</w>', 'the</w>', 'comp', 'any</w>', 'has</w>', 'org', 'ani', 'zed</w>', 'the</w>', 'T', 'H']

Token IDs:
[1146, 1228, 1730, 1103, 1066, 1919, 1272, 78, 999, 1082, 70, 1102, 1875, 1735, 90, 1434, 869, 1927, 1196, 695, 991, 1452, 1389, 1260, 1875, 173

## 4. Token Sequences

The decoder-only Transformer still learns next-token prediction at every context position.

Each WikiText row is encoded independently with the trained BPE tokenizer.

Token sequences are divided into fixed-length windows of `CONTEXT_LENGTH + 1` tokens. The first `CONTEXT_LENGTH` tokens form the input and the same sequence shifted by one position forms the target.

For example:

input tokens  → `[t0, t1, t2, ..., t15]`  
target tokens → `[t1, t2, t3, ..., t16]`

Unlike the character-level examples used in earlier versions, the windows now contain subword token identifiers.

Windows are created independently inside each text row so examples never cross row boundaries. A stride equal to `CONTEXT_LENGTH` avoids excessive overlap and keeps the experiment compact.

In [8]:
def create_token_sequences(texts, tokenizer, context_length):
    inputs, targets = [], []

    for text in texts:
        token_ids = tokenizer.encode(text).ids

        for start in range(0, len(token_ids) - context_length, context_length):
            sequence = token_ids[start:start + context_length + 1]

            if len(sequence) == context_length + 1:
                inputs.append(sequence[:-1])
                targets.append(sequence[1:])

    return tf.constant(inputs, dtype=tf.int32), tf.constant(targets, dtype=tf.int32)

In [9]:
train_inputs, train_targets = create_token_sequences(train_data["text"], tokenizer, CONTEXT_LENGTH)
validation_inputs, validation_targets = create_token_sequences(validation_data["text"], tokenizer, CONTEXT_LENGTH)
test_inputs, test_targets = create_token_sequences(test_data["text"], tokenizer, CONTEXT_LENGTH)

print("Train:", train_inputs.shape, train_targets.shape)
print("Validation:", validation_inputs.shape, validation_targets.shape)
print("Test:", test_inputs.shape, test_targets.shape)

print("\nFirst input IDs:")
print(train_inputs[0].numpy())

print("\nFirst target IDs:")
print(train_targets[0].numpy())

print("\nDecoded input:")
print(tokenizer.decode(train_inputs[0].numpy()))

Train: (96893, 16) (96893, 16)
Validation: (9764, 16) (9764, 16)
Test: (9572, 16) (9572, 16)

First input IDs:
[1146 1228 1730 1103 1066 1919 1272   78  999 1082   70 1102 1875 1735
   90 1434]

First target IDs:
[1228 1730 1103 1066 1919 1272   78  999 1082   70 1102 1875 1735   90
 1434  869]

Decoded input:
In cooperation with various nonprofit organizations


In [10]:
train_inputs = train_inputs[:MAX_TRAIN_SEQUENCES]
train_targets = train_targets[:MAX_TRAIN_SEQUENCES]

validation_inputs = validation_inputs[:MAX_VALIDATION_SEQUENCES]
validation_targets = validation_targets[:MAX_VALIDATION_SEQUENCES]

test_inputs = test_inputs[:MAX_TEST_SEQUENCES]
test_targets = test_targets[:MAX_TEST_SEQUENCES]

## 5. Decoder-Only Transformer

The decoder-only Transformer architecture remains mathematically unchanged from Version 11.

Token embeddings, positional embeddings, two Transformer blocks and the final vocabulary projection keep the same dimensions and forward calculations.

Each Transformer block still contains:

- query, key and value projections;
- causal scaled dot-product self-attention;
- an attention output projection;
- a residual connection followed by LayerNorm;
- a position-wise feed-forward network;
- a second residual connection followed by LayerNorm.

Two independent Transformer blocks are stacked sequentially.

The final projection maps the model representation at every token position to logits over the complete BPE vocabulary.

The only structural change in Version 12 is that `TransformerBlock` and `DecoderOnlyLanguageModel` inherit from `tf.Module`.

This does not change the model mathematics or parameter count. It makes the existing TensorFlow variables explicitly trackable so they can be saved and restored through TensorFlow checkpoints.

In [11]:
def layer_normalize(values, scale, shift):
    mean = tf.reduce_mean(values, axis=-1, keepdims=True)
    variance = tf.reduce_mean(tf.square(values - mean), axis=-1, keepdims=True)
    normalized = (values - mean) / tf.sqrt(variance + LAYER_NORM_EPSILON)
    return normalized * scale + shift


class TransformerBlock(tf.Module):
    def __init__(self, random_generator, name):
        super().__init__(name=name)

        self.query_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02), name="query_weights")
        self.key_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02), name="key_weights")
        self.value_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02), name="value_weights")
        self.attention_output_weights = tf.Variable(random_generator.normal((ATTENTION_DIM, EMBEDDING_DIM), stddev=0.02), name="attention_output_weights")

        self.layer_norm_1_scale = tf.Variable(tf.ones((EMBEDDING_DIM,)), name="layer_norm_1_scale")
        self.layer_norm_1_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,)), name="layer_norm_1_shift")

        self.feed_forward_input_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, FEED_FORWARD_DIM), stddev=0.02), name="feed_forward_input_weights")
        self.feed_forward_output_weights = tf.Variable(random_generator.normal((FEED_FORWARD_DIM, EMBEDDING_DIM), stddev=0.02), name="feed_forward_output_weights")

        self.layer_norm_2_scale = tf.Variable(tf.ones((EMBEDDING_DIM,)), name="layer_norm_2_scale")
        self.layer_norm_2_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,)), name="layer_norm_2_shift")

    @property
    def trainable_variables(self):
        return [
            self.query_weights, self.key_weights, self.value_weights,
            self.attention_output_weights,
            self.layer_norm_1_scale, self.layer_norm_1_shift,
            self.feed_forward_input_weights, self.feed_forward_output_weights,
            self.layer_norm_2_scale, self.layer_norm_2_shift,
        ]

    def __call__(self, values):
        queries = tf.matmul(values, self.query_weights)
        keys = tf.matmul(values, self.key_weights)
        values_projected = tf.matmul(values, self.value_weights)

        scores = tf.matmul(queries, keys, transpose_b=True)
        scores = scores / tf.sqrt(tf.cast(ATTENTION_DIM, tf.float32))

        sequence_length = tf.shape(scores)[-1]
        causal_mask = tf.linalg.band_part(tf.ones((sequence_length, sequence_length)), -1, 0)
        masked_scores = tf.where(causal_mask == 1, scores, tf.constant(-1e9, dtype=tf.float32))

        attention_weights = tf.nn.softmax(masked_scores, axis=-1)
        attention_output = tf.matmul(attention_weights, values_projected)
        attention_output = tf.matmul(attention_output, self.attention_output_weights)

        values = layer_normalize(values + attention_output, self.layer_norm_1_scale, self.layer_norm_1_shift)

        feed_forward = tf.nn.relu(tf.matmul(values, self.feed_forward_input_weights))
        feed_forward = tf.matmul(feed_forward, self.feed_forward_output_weights)

        output = layer_normalize(values + feed_forward, self.layer_norm_2_scale, self.layer_norm_2_shift)

        return output, attention_weights


class DecoderOnlyLanguageModel(tf.Module):
    def __init__(self, vocabulary_size, seed=42, name="decoder_only_language_model"):
        super().__init__(name=name)
        random_generator = tf.random.Generator.from_seed(seed)

        self.token_embeddings = tf.Variable(random_generator.normal((vocabulary_size, EMBEDDING_DIM), stddev=0.02), name="token_embeddings")
        self.positional_embeddings = tf.Variable(random_generator.normal((CONTEXT_LENGTH, EMBEDDING_DIM), stddev=0.02), name="positional_embeddings")

        self.blocks = [
            TransformerBlock(random_generator, name=f"transformer_block_{index + 1}")
            for index in range(NUM_BLOCKS)
        ]

        self.output_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, vocabulary_size), stddev=0.02), name="output_weights")

    @property
    def trainable_variables(self):
        variables = [self.token_embeddings, self.positional_embeddings]

        for block in self.blocks:
            variables.extend(block.trainable_variables)

        variables.append(self.output_weights)
        return variables

    def __call__(self, input_ids):
        token_values = tf.gather(self.token_embeddings, input_ids)
        values = token_values + self.positional_embeddings
        attention_weights = []

        for block in self.blocks:
            values, block_attention = block(values)
            attention_weights.append(block_attention)

        logits = tf.matmul(values, self.output_weights)
        return values, attention_weights, logits

In [12]:
vocabulary_size = tokenizer.get_vocab_size()

model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)

print("Tokenizer vocabulary size:", vocabulary_size)
print("Token embedding shape:", model.token_embeddings.shape)

Tokenizer vocabulary size: 2048
Token embedding shape: (2048, 8)


### Model Structure and Trainable Parameters

The decoder-only Transformer now operates on BPE token sequences instead of character identifiers.

The tokenizer uses a vocabulary of 2048 tokens and every input sequence contains 16 token positions.

The Transformer architecture itself remains unchanged: two independent causal Transformer blocks process the token and positional embeddings before the final projection produces logits over the complete BPE vocabulary.

Inspecting the tensor shapes and parameter count confirms that the new token representation is connected correctly to the model.

In [13]:
example_input = train_inputs[:1]

model_output, attention_weights, logits = model(example_input)

total_parameters = sum(int(np.prod(variable.shape)) for variable in model.trainable_variables)

print("Vocabulary size:", vocabulary_size)
print("Number of Transformer blocks:", len(model.blocks))
print("Number of trainable variables:", len(model.trainable_variables))

print("\nInput shape:", example_input.shape)
print("Model output shape:", model_output.shape)
print("Logits shape:", logits.shape)

for index, block_attention in enumerate(attention_weights, start=1):
    print(f"Block {index} attention shape:", block_attention.shape)

print("\nTotal trainable parameters:", total_parameters)

Vocabulary size: 2048
Number of Transformer blocks: 2
Number of trainable variables: 23

Input shape: (1, 16)
Model output shape: (1, 16, 8)
Logits shape: (1, 16, 2048)
Block 1 attention shape: (1, 16, 16)
Block 2 attention shape: (1, 16, 16)

Total trainable parameters: 34496


## 6. Reusable Training Pipeline

Version 11 trained the model inside one uninterrupted loop.

Version 12 keeps the same next-token prediction objective, sparse categorical cross-entropy, manual gradient-descent update, batch size and learning rate, but separates training into reusable functions.

`make_dataset` creates mini-batches.

`compute_batch_loss` measures next-token prediction loss for one batch.

`evaluate_model` measures a complete dataset without updating parameters.

`train_one_epoch` performs one training epoch.

`run_training_pipeline` will later coordinate epochs, validation and checkpoint decisions.

Training examples use an epoch-specific shuffle seed so resumed training can continue from the correct epoch.

The manual parameter update from Version 11 remains unchanged.

In [14]:
def make_dataset(inputs, targets, batch_size, shuffle_seed=None):
    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))

    if shuffle_seed is not None:
        dataset = dataset.shuffle(buffer_size=len(inputs), seed=shuffle_seed, reshuffle_each_iteration=False)

    return dataset.batch(batch_size)


validation_dataset = make_dataset(validation_inputs, validation_targets, BATCH_SIZE)
test_dataset = make_dataset(test_inputs, test_targets, BATCH_SIZE)

print("Training sequences:", len(train_inputs))
print("Validation sequences:", len(validation_inputs))
print("Test sequences:", len(test_inputs))
print("Batches per training epoch:", int(np.ceil(len(train_inputs) / BATCH_SIZE)))

Training sequences: 12000
Validation sequences: 2000
Test sequences: 2000
Batches per training epoch: 375


In [15]:
def compute_batch_loss(model, inputs, targets):
    _, _, logits = model(inputs)

    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)

    return tf.reduce_mean(losses)

In [16]:
def evaluate_model(model, dataset):
    total_loss = 0.0
    total_tokens = 0

    for batch_inputs, batch_targets in dataset:
        batch_loss = compute_batch_loss(model, batch_inputs, batch_targets)
        batch_tokens = int(tf.size(batch_targets))

        total_loss += float(batch_loss) * batch_tokens
        total_tokens += batch_tokens

    return total_loss / total_tokens

### Gradient Descent

Each mini-batch performs one gradient-descent update.

TensorFlow's `GradientTape` records the operations used to compute the batch loss and obtains the gradient of that loss with respect to every trainable variable.

The parameters are then updated explicitly using the configured learning rate.

The mean training loss is accumulated from the mini-batches processed during each epoch, while validation loss is computed separately without modifying the model parameters.

Token embedding lookups can produce sparse `IndexedSlices` gradients. These are updated with `scatter_sub`, while dense gradients use the standard `assign_sub` update.

In [17]:
def apply_gradient(variable, gradient, learning_rate):
    if gradient is None:
        return

    if isinstance(gradient, tf.IndexedSlices):
        update = tf.IndexedSlices(
            values=learning_rate * gradient.values,
            indices=gradient.indices,
            dense_shape=gradient.dense_shape
        )
        variable.scatter_sub(update)

    else:
        variable.assign_sub(learning_rate * gradient)

In [18]:
def train_one_epoch(model, inputs, targets, epoch, learning_rate):
    training_dataset = make_dataset(
        inputs,
        targets,
        BATCH_SIZE,
        shuffle_seed=SEED + epoch,
    )

    total_loss = 0.0
    total_tokens = 0

    for batch_inputs, batch_targets in training_dataset:
        with tf.GradientTape() as tape:
            batch_loss = compute_batch_loss(model, batch_inputs, batch_targets)

        gradients = tape.gradient(batch_loss, model.trainable_variables)

        for variable, gradient in zip(model.trainable_variables, gradients):
            apply_gradient(variable, gradient, learning_rate)

        batch_tokens = int(tf.size(batch_targets))
        total_loss += float(batch_loss) * batch_tokens
        total_tokens += batch_tokens

    return total_loss / total_tokens

## 7. Training State and Checkpoints

Version 12 introduces the minimum training state required to resume an interrupted run.

Three values are tracked:

- `completed_epoch`, the most recently completed epoch;
- `best_epoch`, the epoch with the lowest validation loss;
- `best_validation_loss`, the lowest validation loss observed so far.

Two checkpoints serve different purposes:

- the **last checkpoint** stores the current model and training state so training can resume;
- the **best checkpoint** stores the model selected by validation loss.

The runtime checkpoint directory starts clean when the notebook is executed.

Training will first stop at epoch 10. A new model object will then restore the last checkpoint and continue from epoch 11 through epoch 20.

After training, another new model object will restore the best checkpoint for final evaluation.

In [19]:
RUNTIME_DIRECTORY = Path("/kaggle/working/runtime-v12")

if RUNTIME_DIRECTORY.exists():
    shutil.rmtree(RUNTIME_DIRECTORY)

RUNTIME_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Runtime checkpoints:", RUNTIME_DIRECTORY)

Runtime checkpoints: /kaggle/working/runtime-v12


In [20]:
def build_checkpoint_managers(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    runtime_directory,
):
    last_checkpoint = tf.train.Checkpoint(
        model=model,
        completed_epoch=completed_epoch,
        best_epoch=best_epoch,
        best_validation_loss=best_validation_loss,
    )

    best_checkpoint = tf.train.Checkpoint(
        model=model,
        best_epoch=best_epoch,
        best_validation_loss=best_validation_loss,
    )

    last_manager = tf.train.CheckpointManager(
        last_checkpoint,
        str(runtime_directory / "last"),
        max_to_keep=1,
    )

    best_manager = tf.train.CheckpointManager(
        best_checkpoint,
        str(runtime_directory / "best"),
        max_to_keep=1,
    )

    return last_checkpoint, best_checkpoint, last_manager, best_manager

In [21]:
def run_training_pipeline(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    train_inputs,
    train_targets,
    validation_dataset,
    final_epoch,
    learning_rate,
    last_manager,
    best_manager,
):
    start_epoch = int(completed_epoch.numpy()) + 1

    for epoch in range(start_epoch, final_epoch + 1):
        training_loss = train_one_epoch(
            model,
            train_inputs,
            train_targets,
            epoch,
            learning_rate,
        )

        validation_loss = evaluate_model(model, validation_dataset)

        completed_epoch.assign(epoch)

        if validation_loss < float(best_validation_loss.numpy()):
            best_validation_loss.assign(validation_loss)
            best_epoch.assign(epoch)
            best_manager.save(checkpoint_number=epoch)

        last_manager.save(checkpoint_number=epoch)

        if epoch == 1 or epoch % 10 == 0 or epoch == final_epoch:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {training_loss:.6f} | "
                f"Validation loss: {validation_loss:.6f}"
            )

In [22]:
completed_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

last_checkpoint, best_checkpoint, last_manager, best_manager = build_checkpoint_managers(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    RUNTIME_DIRECTORY,
)

initial_train_dataset = make_dataset(train_inputs, train_targets, BATCH_SIZE)

initial_train_loss = evaluate_model(model, initial_train_dataset)
initial_validation_loss = evaluate_model(model, validation_dataset)
random_baseline = float(np.log(vocabulary_size))

print(f"Initial training loss: {initial_train_loss:.6f}")
print(f"Initial validation loss: {initial_validation_loss:.6f}")
print(f"Random baseline: {random_baseline:.6f}")

Initial training loss: 7.626330
Initial validation loss: 7.625952
Random baseline: 7.624619


## 8. Simulated Interruption, Restoration and Resume

Training first proceeds to `INTERRUPT_AFTER_EPOCH`, which is epoch 10.

At that point, the last checkpoint contains the model parameters and the training state required to continue.

A new model object and new state variables are then created to simulate a fresh training process.

The last checkpoint is restored into these new objects.

Because the restored `completed_epoch` is 10, training resumes from epoch 11 and continues until `TOTAL_EPOCHS`, which is epoch 20.

The best checkpoint is updated whenever validation loss reaches a new minimum.

This demonstrates the two roles introduced in Version 12:

- the last checkpoint determines **where training resumes**;
- the best checkpoint determines **which model is selected**.

In [23]:
run_training_pipeline(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    train_inputs,
    train_targets,
    validation_dataset,
    INTERRUPT_AFTER_EPOCH,
    LEARNING_RATE,
    last_manager,
    best_manager,
)

print("\nInterrupted after epoch:", int(completed_epoch.numpy()))
print("Last checkpoint:", last_manager.latest_checkpoint)
print("Best checkpoint:", best_manager.latest_checkpoint)

Epoch   1 | Train loss: 6.030380 | Validation loss: 5.521137
Epoch  10 | Train loss: 5.068344 | Validation loss: 5.108189

Interrupted after epoch: 10
Last checkpoint: /kaggle/working/runtime-v12/last/ckpt-10
Best checkpoint: /kaggle/working/runtime-v12/best/ckpt-10


In [24]:
resumed_model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)
_ = resumed_model(example_input)

resumed_completed_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
resumed_best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
resumed_best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

(
    resumed_last_checkpoint,
    resumed_best_checkpoint,
    resumed_last_manager,
    resumed_best_manager,
) = build_checkpoint_managers(
    resumed_model,
    resumed_completed_epoch,
    resumed_best_epoch,
    resumed_best_validation_loss,
    RUNTIME_DIRECTORY,
)

if resumed_last_manager.latest_checkpoint is None:
    raise RuntimeError("Last checkpoint not found.")

resumed_last_checkpoint.restore(
    resumed_last_manager.latest_checkpoint
).assert_existing_objects_matched()

restored_epoch = int(resumed_completed_epoch.numpy())

print("Restored completed epoch:", restored_epoch)
print("Resume begins at epoch:", restored_epoch + 1)

Restored completed epoch: 10
Resume begins at epoch: 11


In [25]:
run_training_pipeline(
    resumed_model,
    resumed_completed_epoch,
    resumed_best_epoch,
    resumed_best_validation_loss,
    train_inputs,
    train_targets,
    validation_dataset,
    TOTAL_EPOCHS,
    LEARNING_RATE,
    resumed_last_manager,
    resumed_best_manager,
)

print("\nCompleted epoch:", int(resumed_completed_epoch.numpy()))
print("Best validation epoch:", int(resumed_best_epoch.numpy()))
print("Best validation loss:", f"{float(resumed_best_validation_loss.numpy()):.6f}")

Epoch  20 | Train loss: 5.015265 | Validation loss: 5.082863

Completed epoch: 20
Best validation epoch: 18
Best validation loss: 5.074273


## 9. Restore the Best Model for Final Evaluation

The last checkpoint determines where training resumes.

The best checkpoint determines which observed model should be selected.

After the 20-epoch training budget is complete, a new model instance is created and restored from the checkpoint with the lowest validation loss.

Training and validation losses are recomputed using this restored model.

Only after model selection is complete is the independent test split evaluated.

The test split therefore does not influence training or checkpoint selection.

In [26]:
final_model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)
_ = final_model(example_input)

selected_best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
selected_best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

final_best_checkpoint = tf.train.Checkpoint(
    model=final_model,
    best_epoch=selected_best_epoch,
    best_validation_loss=selected_best_validation_loss,
)

if resumed_best_manager.latest_checkpoint is None:
    raise RuntimeError("Best checkpoint not found.")

final_best_checkpoint.restore(
    resumed_best_manager.latest_checkpoint
).assert_existing_objects_matched()

final_train_loss = evaluate_model(
    final_model,
    make_dataset(train_inputs, train_targets, BATCH_SIZE),
)

final_validation_loss = evaluate_model(final_model, validation_dataset)
test_loss = evaluate_model(final_model, test_dataset)

print("Selected best epoch:", int(selected_best_epoch.numpy()))
print(f"Final training loss: {final_train_loss:.6f}")
print(f"Final validation loss: {final_validation_loss:.6f}")
print(f"Test loss: {test_loss:.6f}")
print(f"Random baseline: {random_baseline:.6f}")

Selected best epoch: 18
Final training loss: 4.986566
Final validation loss: 5.074273
Test loss: 5.105811
Random baseline: 7.624619


## 10. Next-Token Prediction

The restored best decoder-only Transformer produces one probability distribution over the BPE vocabulary at every context position.

For next-token prediction, the model receives a sequence of 16 tokens and the logits produced at the final position represent the prediction for the token that should follow the complete context.

Applying softmax converts these logits into probabilities.

Because the vocabulary contains BPE subword units, the predictions may represent complete words, word fragments or punctuation rather than individual characters.

Version 12 performs this inspection using `final_model`, the independently reconstructed model restored from the best validation checkpoint.

In [27]:
def predict_next_token_probabilities(model, input_ids):
    input_tensor = tf.constant([input_ids], dtype=tf.int32)

    _, _, logits = model(input_tensor)

    next_token_logits = logits[0, -1]
    probabilities = tf.nn.softmax(next_token_logits)

    return probabilities.numpy()

In [28]:
prediction_context = test_inputs[0].numpy()

print("Context token IDs:")
print(prediction_context)

print("\nContext tokens:")
print([tokenizer.id_to_token(int(token_id)) for token_id in prediction_context])

print("\nDecoded context:")
print(tokenizer.decode(prediction_context.tolist()))

Context token IDs:
[  44 1012 1708   39 1266 1124 1047  991   70 1217 1080 1003 1968 1071
   69  682]

Context tokens:
['L', 'it', 'tle</w>', 'G', 'id', 'ding</w>', 'is</w>', 'the</w>', 'f', 'our', 'th</w>', 'and</w>', 'final</w>', 'po', 'e', 'm</w>']

Decoded context:
Little Gidding is the fourth and final poem


In [29]:
probabilities = predict_next_token_probabilities(final_model, prediction_context)

top_indices = np.argsort(probabilities)[-10:][::-1]

print("Top next-token predictions:\n")

for token_id in top_indices:
    token = tokenizer.id_to_token(int(token_id))
    probability = probabilities[token_id]

    print(
        f"{token!r:20s} "
        f"id={token_id:4d} "
        f"probability={probability:.4f}"
    )

Top next-token predictions:

',</w>'              id= 869 probability=0.1380
'.</w>'              id= 723 probability=0.0794
'and</w>'            id=1003 probability=0.0469
'in</w>'             id=1007 probability=0.0345
'of</w>'             id=1004 probability=0.0289
'was</w>'            id=1029 probability=0.0258
"'</w>"              id= 746 probability=0.0256
'@-@</w>'            id=1040 probability=0.0205
'"</w>'              id= 902 probability=0.0155
'(</w>'              id= 779 probability=0.0144


In [30]:
true_next_token_id = int(test_targets[0, -1])
true_next_token = tokenizer.id_to_token(true_next_token_id)

print("True next token:", repr(true_next_token))
print("True token ID:", true_next_token_id)
print("Model probability:", f"{probabilities[true_next_token_id]:.4f}")

True next token: 'of</w>'
True token ID: 1004
Model probability: 0.0289


## 11. Autoregressive Generation

The restored best model can now generate text one BPE token at a time.

Generation begins from an initial text prompt. The prompt is encoded with the same BPE tokenizer used during training.

At every step:

1. the most recent `CONTEXT_LENGTH` token identifiers are passed to the Transformer;
2. the logits at the final position are converted into probabilities;
3. the token with the highest probability is selected;
4. the selected token is appended to the sequence;
5. the process repeats using the updated context.

Version 12 keeps the same greedy decoding procedure used in Version 11, selecting the most probable token at every step.

Because the model has a context length of 16 tokens, the prompt must contain at least 16 BPE tokens. Once the sequence becomes longer, only the most recent 16 tokens are used as context.

The complete sequence of BPE token identifiers is then decoded back into readable text.

Generation uses `final_model`, restored from the best validation checkpoint, so the generated output comes from the same model selected for final evaluation.

In [31]:
def generate_text(model, tokenizer, prompt, number_of_tokens):
    generated_ids = tokenizer.encode(prompt).ids

    if len(generated_ids) < CONTEXT_LENGTH:
        raise ValueError(f"The prompt must contain at least {CONTEXT_LENGTH} tokens.")

    for _ in range(number_of_tokens):
        context_ids = generated_ids[-CONTEXT_LENGTH:]

        probabilities = predict_next_token_probabilities(model, context_ids)

        next_token_id = int(np.argmax(probabilities))
        generated_ids.append(next_token_id)

    return generated_ids, tokenizer.decode(generated_ids)

In [32]:
generation_prompt = tokenizer.decode(prediction_context.tolist())

generated_ids, generated_text = generate_text(
    final_model,
    tokenizer,
    generation_prompt,
    number_of_tokens=40,
)

print("Prompt:")
print(generation_prompt)

print("\nGenerated text:")
print(generated_text)

Prompt:
Little Gidding is the fourth and final poem

Generated text:
Little Gidding is the fourth and final poem , and the Sess , and the Sess , and the Sess , and the Sess , and the Sess , and the Se


In [33]:
prompt_length = len(tokenizer.encode(generation_prompt).ids)

new_token_ids = generated_ids[prompt_length:]

print("Generated BPE tokens:")
print([tokenizer.id_to_token(token_id) for token_id in new_token_ids])

Generated BPE tokens:
[',</w>', 'and</w>', 'the</w>', 'S', 'e', 's', 's</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e', 's', 's</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e', 's', 's</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e', 's', 's</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e', 's', 's</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e']


## 12. Inspecting Attention

The decoder-only Transformer still uses causal self-attention after the transition to BPE tokens.

With a context length of 16, each Transformer block produces a `16 × 16` attention matrix for every example.

The following inspection focuses on the final context position used for next-token prediction.

The attention weights show how strongly the final token attends to each earlier BPE token in the context.

Because the attention mechanism is causal, the final position may attend to every token in the current context, while earlier positions cannot attend to future tokens.

Version 12 performs this inspection on `final_model`, restored from the best validation checkpoint, so the attention patterns belong to the same model used for final evaluation and generation.

In [34]:
attention_context = test_inputs[0:1]

_, attention_weights, _ = final_model(attention_context)

attention_tokens = [
    tokenizer.id_to_token(int(token_id))
    for token_id in attention_context[0].numpy()
]

print("Context:")
print(tokenizer.decode(attention_context[0].numpy().tolist()))

print("\nTokens:")
print(attention_tokens)

Context:
Little Gidding is the fourth and final poem

Tokens:
['L', 'it', 'tle</w>', 'G', 'id', 'ding</w>', 'is</w>', 'the</w>', 'f', 'our', 'th</w>', 'and</w>', 'final</w>', 'po', 'e', 'm</w>']


In [35]:
for block_index, block_attention in enumerate(attention_weights, start=1):
    final_position_attention = block_attention[0, -1].numpy()

    print(f"\nTransformer block {block_index}")
    print("-" * 45)

    for token, weight in zip(attention_tokens, final_position_attention):
        print(f"{token!r:18s} {weight:.4f}")


Transformer block 1
---------------------------------------------
'L'                0.0624
'it'               0.0624
'tle</w>'          0.0625
'G'                0.0624
'id'               0.0625
'ding</w>'         0.0625
'is</w>'           0.0625
'the</w>'          0.0627
'f'                0.0624
'our'              0.0625
'th</w>'           0.0625
'and</w>'          0.0626
'final</w>'        0.0625
'po'               0.0625
'e'                0.0625
'm</w>'            0.0625

Transformer block 2
---------------------------------------------
'L'                0.2162
'it'               0.0284
'tle</w>'          0.0201
'G'                0.3228
'id'               0.0320
'ding</w>'         0.0154
'is</w>'           0.0148
'the</w>'          0.0520
'f'                0.1224
'our'              0.0259
'th</w>'           0.0234
'and</w>'          0.0256
'final</w>'        0.0171
'po'               0.0224
'e'                0.0327
'm</w>'            0.0288


In [36]:
for block_index, block_attention in enumerate(attention_weights, start=1):
    final_position_attention = block_attention[0, -1].numpy()

    print(f"Block {block_index} attention sum:", f"{final_position_attention.sum():.6f}")

Block 1 attention sum: 1.000000
Block 2 attention sum: 1.000000


## 13. Results

Version 12 keeps the Version 11 language-model experiment and adds checkpoint-based training state.

Training is intentionally interrupted after epoch 10.

A newly constructed model restores the last checkpoint and resumes from epoch 11 through the original 20-epoch training budget.

During training, validation loss determines which model is stored as the best checkpoint.

After training, another new model instance restores that best checkpoint before final evaluation.

The complete Version 12 training flow is therefore:

training  
→ last checkpoint  
→ interruption  
→ restore and resume  
→ best checkpoint  
→ restored final model  
→ final evaluation

Next-token prediction, autoregressive generation and attention inspection all use the same restored best model.

## 14. Tests

The following checks verify the main structural, numerical and checkpoint properties of Version 12.

They confirm that sequence shapes and model dimensions remain correct, losses are finite, causal attention remains normalized and masked, checkpoints are created, training resumes from epoch 10, the full 20-epoch budget is completed, and the final model is restored from the best validation checkpoint.

The tests also verify next-token probabilities and autoregressive generation.

In [37]:
assert vocabulary_size == VOCABULARY_SIZE
assert tokenizer.token_to_id("[UNK]") is not None

assert train_inputs.shape == (MAX_TRAIN_SEQUENCES, CONTEXT_LENGTH)
assert train_targets.shape == (MAX_TRAIN_SEQUENCES, CONTEXT_LENGTH)
assert validation_inputs.shape == (MAX_VALIDATION_SEQUENCES, CONTEXT_LENGTH)
assert validation_targets.shape == (MAX_VALIDATION_SEQUENCES, CONTEXT_LENGTH)
assert test_inputs.shape == (MAX_TEST_SEQUENCES, CONTEXT_LENGTH)
assert test_targets.shape == (MAX_TEST_SEQUENCES, CONTEXT_LENGTH)

assert np.array_equal(
    train_inputs[0, 1:].numpy(),
    train_targets[0, :-1].numpy(),
)

assert model_output.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)
assert logits.shape == (1, CONTEXT_LENGTH, vocabulary_size)
assert len(model.blocks) == NUM_BLOCKS
assert model.blocks[0] is not model.blocks[1]
assert len(model.trainable_variables) == 23
assert total_parameters == 34496

assert last_manager.latest_checkpoint is not None
assert best_manager.latest_checkpoint is not None
assert resumed_last_manager.latest_checkpoint is not None
assert resumed_best_manager.latest_checkpoint is not None

assert restored_epoch == INTERRUPT_AFTER_EPOCH
assert int(resumed_completed_epoch.numpy()) == TOTAL_EPOCHS

assert int(selected_best_epoch.numpy()) == int(resumed_best_epoch.numpy())
assert np.isclose(
    float(selected_best_validation_loss.numpy()),
    float(resumed_best_validation_loss.numpy()),
    atol=1e-6,
)

loss_values = [
    initial_train_loss,
    initial_validation_loss,
    final_train_loss,
    final_validation_loss,
    test_loss,
]

assert np.all(np.isfinite(loss_values))

assert final_train_loss < initial_train_loss
assert final_validation_loss < initial_validation_loss
assert test_loss < random_baseline

assert np.isclose(
    final_validation_loss,
    float(selected_best_validation_loss.numpy()),
    atol=1e-5,
)

for block_attention in attention_weights:
    matrix = block_attention[0].numpy()

    assert matrix.shape == (CONTEXT_LENGTH, CONTEXT_LENGTH)
    assert np.allclose(matrix.sum(axis=-1), 1.0, atol=1e-6)
    assert np.allclose(np.triu(matrix, k=1), 0.0, atol=1e-7)

assert np.all(np.isfinite(probabilities))
assert np.isclose(probabilities.sum(), 1.0, atol=1e-6)

assert len(new_token_ids) == 40

print("All checks passed.")

All checks passed.


## Notes

- Version 12 is Version 11 plus training-state and checkpoint management.

- WikiText-103 Raw v1, the BPE tokenizer, sequence limits, model dimensions, learning rate, batch size and 20-epoch training budget remain unchanged.

- The last checkpoint stores the model and the minimum state required to resume training.

- Training is intentionally interrupted after epoch 10 and restored into a newly constructed model before continuing to epoch 20.

- The best checkpoint is selected only by validation loss.

- The test split is evaluated only after the best checkpoint has been restored into `final_model`.

- Next-token prediction, autoregressive generation and attention inspection all use the same restored best model.

- The runtime checkpoint directory is recreated at the beginning of each notebook execution. Persistent checkpoint storage across completely separate Kaggle sessions is intentionally outside the scope of this educational version.

- Generated text remains limited by the deliberately small model, short context window, restricted training subset and greedy decoding strategy.

- Because whitespace pre-tokenization treats punctuation as separate units, decoded text may contain spaces before punctuation.

Future versions will introduce new components and gradually evolve the architecture.